# DBRepo Upload Notebook

This notebook handles the creation of the database, tables, and the upload of the data to DBRepo via the REST API. It ensures proper metadata attribution to Eurostat and applies the CC BY 4.0 license.

**Note:** As per the plan, this notebook sets up the logic but data changes should only be executed once fully approved.

In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [43]:
import os
import glob
import json
import pandas as pd
import requests
import eurostat
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient

load_dotenv()

DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
DATABASE_ID = "123289f2-5218-4b32-b962-5f3dafec1fe3"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

# T2.1 DBRepo - schema design 

In [ ]:
# Metadata for the Database based on the Data Management Plan (DMP) and instructions
database_metadata = {
    "name": "ev_protection_investment_analysis",
    "description": "Environmental protection investments across EU countries (2014-2022) based on the reuse of existing data from the Eurostat Open Data Portal.",
    "publisher": "Eurostat",
    "creator": "Eurostat",
    "license": "CC BY 4.0",
    "rights": "European Union / Eurostat",
    "republisher": "Luka Premuš / TU Wien",
    "republisher_email": "e12552143@student.tuwien.ac.at",
    "republisher_affiliation": "TU Wien (Course 194.045 Data Stewardship)",
    "project_title": "Do Rich Countries Invest More in Saving the Planet? Analysis of Europe’s Green Investment Landscape",
    "dmp_version": "1.0",
    "dmp_date": "2026-05-25"
}
print("Database metadata prepared:", json.dumps(database_metadata, indent=2))

In [ ]:


table_country = {
    "name": "Country",
    "is_public": True,
    "is_schema_public": True,
    "description": "Country dimension table with ISO codes.",
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "country_name", "type": "varchar", "size": 255, "null_allowed": False, "description": "Full name of the country"}
    ],
    "constraints": {
        "primary_key": ["country_code"]
    }
}

table_activity = {
    "name": "Environmental_Activity",
    "is_public": True,
    "is_schema_public": True,
    "description": "Environmental activity dimension table (CEPA/CReMA classifications).",
    "columns": [
        {"name": "ceparema_code", "type": "varchar", "size": 50, "null_allowed": False, "description": "CEPA/CReMA activity code"},
        {"name": "activity_name", "type": "varchar", "size": 255, "null_allowed": False, "description": "Name of the environmental protection activity"}
    ],
    "constraints": {
        "primary_key": ["ceparema_code"]
    }
}


In [ ]:
table_macro = {
    "name": "Macroeconomic_Indicator",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "year", "type": "int", "null_allowed": False, "description": "Observation year"},
        {"name": "population", "type": "bigint", "null_allowed": True, "description": "Total population"},
        {"name": "gdp_per_capita", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Gross Domestic Product per capita"}
    ],
    "constraints": {
        "primary_key": ["country_code", "year"],
        "foreign_keys": [
            {
                "columns": ["country_code"],
                "referenced_table": "country",
                "referenced_columns": ["country_code"]
            }
        ]
    }
}

table_invest = {
    "name": "Environmental_Investment",
    "is_public": True,
    "is_schema_public": True,
    "columns": [
        {"name": "country_code", "type": "varchar", "size": 2, "null_allowed": False, "description": "2-letter ISO country code"},
        {"name": "year", "type": "int", "null_allowed": False, "description": "Observation year"},
        {"name": "ceparema_code", "type": "varchar", "size": 50, "null_allowed": False, "description": "CEPA/CReMA activity code"},
        {"name": "inv_gov", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by general government"},
        {"name": "inv_corp_spec", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by specialist producers"},
        {"name": "inv_corp_anc", "type": "decimal", "size": 15, "d": 2, "null_allowed": True, "description": "Investment by ancillary producers"},
    ],
    "constraints": {
        "primary_key": ["country_code", "year", "ceparema_code"],
        "foreign_keys": [
            {
                "columns": ["country_code"],
                "referenced_table": "country",
                "referenced_columns": ["country_code"]
            },
            {
                "columns": ["ceparema_code"],
                "referenced_table": "environmental_activity",
                "referenced_columns": ["ceparema_code"]
            }
        ]
    }
}


In [ ]:
def create_table_via_api(database_id, table_def):
    # Ensure all constraint fields are present to avoid 500 errors
    if "constraints" not in table_def:
        table_def["constraints"] = {}
    
    for key in ["uniques", "checks", "foreign_keys", "primary_key"]:
        if key not in table_def["constraints"]:
            table_def["constraints"][key] = []

    url = f"{DBREPO_ENDPOINT}/api/v1/database/{database_id}/table"
    print(f"Creating table {table_def['name']}...")
    response = requests.post(url, json=table_def, auth=(USERNAME, PASSWORD))
    
    if response.status_code == 201:
        print(f"Success: Table {table_def['name']} created.")
    elif response.status_code == 409:
        print(f"Warning: Table {table_def['name']} already exists.")
    else:
        print(f"Error ({response.status_code}): {response.text}")


tables_to_create = [table_country, table_activity, table_macro, table_invest]
for t in tables_to_create:
    create_table_via_api(DATABASE_ID, t)


# T2.2 Semantic mapping

In [ ]:
# Code Placeholder

# T2.3 Mapping Units of Measurement

### Ontology Choice

All numeric attributes are mapped using QUDT (Quantities, Units, Dimensions and Types), version 3.2.1 — https://qudt.org

The assignment recommends the SI Digital Framework as the primary ontology, but SI only covers physical quantities. None of the attributes in this dataset are physical measurements: year is a calendar unit, population is a count of persons and all monetary values are in Euro. QUDT is a well-established, actively maintained ontology that explicitly covers all three of these cases and provides more expressive modeling for this case than SI-oriented unit systems.

### Unit Mappings

The attribute year (both tables) is mapped to the QUDT unit Year (https://qudt.org/vocab/unit/YR). QUDT defines this as one passage of Earth around the Sun (roughly 365 days).

The attribute population is mapped to unit NUM (https://qudt.org/vocab/unit/NUM). Since population is a simple count of persons rather than a physical measurement, SI does not provide a meaningful unit.

All monetary attributes (gdp_per_capita, inv_gov, inv_corp_spec, inv_corp_anc, inv_corp_total, inv_total) are mapped to Euro (https://qudt.org/vocab/unit/CCY_EUR). QUDT defines this as a currency unit. Currency values are outside the scope of SI units, which is why a dedicated currency ontology unit is used.

In [50]:
def apply_unit_mappings(base_url, database_id, table_id, unit_mappings, auth):
    """
    Apply QUDT unit mappings to columns of a DBRepo table.

    Parameters
    ----------
    base_url : str
        DBRepo base URL
    database_id : str
        Database ID
    table_id : str
        Table ID
    unit_mappings : dict
        Mapping: column_name -> unit_uri
    auth : tuple
        (USERNAME, PASSWORD)
    """

    # fetch table schema
    url = f"{base_url}/api/v1/database/{database_id}/table/{table_id}"

    r = requests.get(url, auth=auth)
    r.raise_for_status()

    table = r.json()

    # apply mappings
    for col in table["columns"]:
        column_name = col["name"]

        if column_name in unit_mappings:
            column_id = col["id"]
            unit_uri = unit_mappings[column_name]

            update_url = (
                f"{base_url}/api/v1/database/{database_id}"
                f"/table/{table_id}/column/{column_id}")

            response = requests.put(
                update_url,
                json={"unit_uri": unit_uri},
                auth=auth)

            print(
                f"{column_name} -> {unit_uri} "
                f"[status={response.status_code}]")

In [51]:
UNIT_YEAR  = "https://qudt.org/vocab/unit/YR"
UNIT_COUNT = "https://qudt.org/vocab/unit/NUM"
UNIT_EUR   = "https://qudt.org/vocab/unit/CCY_EUR"

In [52]:
# Macroeconomic_Indicator
apply_unit_mappings(
    base_url=DBREPO_ENDPOINT,
    database_id=DATABASE_ID,
    table_id="ca99f3b3-dd67-4df7-9505-5ec0f7a922a6",
    unit_mappings={
        "year": UNIT_YEAR,
        "population": UNIT_COUNT,
        "gdp_per_capita": UNIT_EUR,
    },
    auth=(USERNAME, PASSWORD)
)

year -> https://qudt.org/vocab/unit/YR [status=202]
population -> https://qudt.org/vocab/unit/NUM [status=202]
gdp_per_capita -> https://qudt.org/vocab/unit/CCY_EUR [status=202]


In [53]:
# Environmental_Investment
apply_unit_mappings(
    base_url=DBREPO_ENDPOINT,
    database_id=DATABASE_ID,
    table_id="4f9588d7-45f1-432e-91b9-c9ae5d0fb22c",
    unit_mappings={
        "year": UNIT_YEAR,
        "inv_gov": UNIT_EUR,
        "inv_corp_spec": UNIT_EUR,
        "inv_corp_anc": UNIT_EUR,
        "inv_corp_total": UNIT_EUR,
        "inv_total": UNIT_EUR,
    },
    auth=(USERNAME, PASSWORD)
)

year -> https://qudt.org/vocab/unit/YR [status=202]
inv_gov -> https://qudt.org/vocab/unit/CCY_EUR [status=202]
inv_corp_spec -> https://qudt.org/vocab/unit/CCY_EUR [status=202]
inv_corp_anc -> https://qudt.org/vocab/unit/CCY_EUR [status=202]


**Note:** 
The REST API via the requests library is used instead of the DBRepo Python client because the current DBRepo SDK does not fully support the latest column metadata fields, especially unit_uri and concept_uri. In the SDK, these fields are either missing or not correctly mapped in the data models, which leads to incomplete or inconsistent results when reading the table structure.

With the REST API, the updates are visible and correctly stored in the backend. A direct GET request to the table endpoint shows that the unit_uri values are correctly set for the corresponding columns.

However, these changes are currently not reflected in the frontend interface. The frontend schema view still does not display the updated measurement units, even though the REST API confirms that the values are present in the database. This indicates that the backend is up to date, but the frontend either uses cached data or does not yet render the unit_uri field in the schema view.


In [54]:
# Example
TABLE_ID = "ca99f3b3-dd67-4df7-9505-5ec0f7a922a6"

url = f"{DBREPO_ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{TABLE_ID}"

r = requests.get(url, auth=(USERNAME, PASSWORD))
r.raise_for_status()
table = r.json()

print(f"TABLE: {table['name']} ({table['id']})")

for col in table["columns"]:
    print("--------------------------------------------------")
    print(f"Column name      : {col.get('name')}")
    print(f"Type             : {col.get('type')}")
    print(f"Unit URI         : {col.get('unit_uri')}")

TABLE: Macroeconomic_Indicator (ca99f3b3-dd67-4df7-9505-5ec0f7a922a6)
--------------------------------------------------
Column name      : country_code
Type             : varchar
Unit URI         : None
--------------------------------------------------
Column name      : year
Type             : int
Unit URI         : https://qudt.org/vocab/unit/YR
--------------------------------------------------
Column name      : population
Type             : bigint
Unit URI         : https://qudt.org/vocab/unit/NUM
--------------------------------------------------
Column name      : gdp_per_capita
Type             : decimal
Unit URI         : https://qudt.org/vocab/unit/CCY_EUR


In [55]:
TABLE_ID = "ca99f3b3-dd67-4df7-9505-5ec0f7a922a6"

table = client.get_table(DATABASE_ID, TABLE_ID)

print(f"TABLE: {table.name} ({table.id})\n")

for col in table.columns:
    print("--------------------------------------------------")
    print(f"Column name      : {col.name}")
    print(f"unit             : {getattr(col, 'unit', None)}")
    print(f"unit_uri         : {getattr(col, 'unit_uri', None)}")

TABLE: Macroeconomic_Indicator (ca99f3b3-dd67-4df7-9505-5ec0f7a922a6)

--------------------------------------------------
Column name      : country_code
unit             : None
unit_uri         : None
--------------------------------------------------
Column name      : year
unit             : None
unit_uri         : None
--------------------------------------------------
Column name      : population
unit             : None
unit_uri         : None
--------------------------------------------------
Column name      : gdp_per_capita
unit             : None
unit_uri         : None


# T2.5 DBRepo - load

In [44]:
# Load raw data
raw_dir = os.path.join("..", "data", "raw")

# Safely find and load the latest raw files using the new variable names
df_pop_raw = pd.read_csv(sorted(glob.glob(os.path.join(raw_dir, "*_raw_demo_pjan.csv")))[-1])
df_gdp_raw = pd.read_csv(sorted(glob.glob(os.path.join(raw_dir, "*_raw_sdg_08_10.csv")))[-1])
df_gov_raw = pd.read_csv(sorted(glob.glob(os.path.join(raw_dir, "*_raw_env_ac_epigg1.csv")))[-1])
df_corp_spec_raw = pd.read_csv(sorted(glob.glob(os.path.join(raw_dir, "*_raw_env_ac_epissp1.csv")))[-1])
df_corp_anc_raw = pd.read_csv(sorted(glob.glob(os.path.join(raw_dir, "*_raw_env_ac_epiap1.csv")))[-1])

# Standardize the 'geo' column name across all raw files
for df in [df_pop_raw, df_gdp_raw, df_gov_raw, df_corp_spec_raw, df_corp_anc_raw]:
    df.rename(columns={'geo\\TIME_PERIOD': 'geo'}, inplace=True)

In [45]:
# Define reusable melt & filter function
def process_raw_df(df, unit_filter, value_name):
    # Filter by required unit
    df_clean = df[df['unit'] == unit_filter].copy()
    
    # Remove EU regional aggregates
    aggregates = ['EU27_2020', 'EU28', 'EA19', 'EA20']
    df_clean = df_clean[~df_clean['geo'].isin(aggregates)]
    
    # Identify year columns between 2014 and 2022
    year_cols = [c for c in df_clean.columns if str(c).isdigit() and 2014 <= int(c) <= 2022]
    id_vars = [c for c in df_clean.columns if not str(c).isdigit()]
    
    # Melt to long format
    df_long = df_clean.melt(
        id_vars=id_vars, value_vars=year_cols, var_name='year', value_name=value_name
    )
    
    # Convert types
    df_long['year'] = df_long['year'].astype(int)
    df_long[value_name] = pd.to_numeric(df_long[value_name], errors='coerce')
    
    return df_long

In [46]:
# Preprocess Macroeconomic data (fact table)

# Population (NR, sex=T, age=TOTAL)
df_pop_filtered = df_pop_raw[(df_pop_raw['sex'] == 'T') & (df_pop_raw['age'] == 'TOTAL')]
df_pop = process_raw_df(df_pop_filtered, 'NR', 'population')

# GDP (CLV20_EUR_HAB, na_item=B1GQ)
df_gdp_filtered = df_gdp_raw[df_gdp_raw['na_item'] == 'B1GQ']
df_gdp = process_raw_df(df_gdp_filtered, 'CLV20_EUR_HAB', 'gdp_per_capita')

# Merge into Macroeconomic Fact Table
df_macro = pd.merge(
    df_pop[['geo', 'year', 'population']],
    df_gdp[['geo', 'year', 'gdp_per_capita']],
    on=['geo', 'year'], how='outer'
)
df_macro.rename(columns={'geo': 'country_code'}, inplace=True)
df_macro.dropna(subset=['country_code', 'year'], inplace=True)

df_macro.head(5)

,country_code,year,population,gdp_per_capita
0,AD,2014,NaN,NaN
1,AD,2015,NaN,NaN
2,AD,2016,NaN,NaN
3,AD,2017,NaN,NaN
4,AD,2018,NaN,NaN


In [47]:
# Preprocess Investment data (fact table)

# Government & Corporate Specialists
df_gov = process_raw_df(df_gov_raw, 'MIO_EUR', 'inv_gov')
df_corp_spec = process_raw_df(df_corp_spec_raw, 'MIO_EUR', 'inv_corp_spec')

# Corporate Ancillary (Requires specific filtering before melting)
df_corp_anc_filtered = df_corp_anc_raw[
    (df_corp_anc_raw['nace_r2'] == 'EP_BSN') & 
    (df_corp_anc_raw['env_econ'].isin(['EPS_INV_PT', 'EPS_INV_PP']))
]
df_corp_anc = process_raw_df(df_corp_anc_filtered, 'MIO_EUR', 'inv_corp_anc')

# Group and sum the Ancillary data
df_corp_anc = df_corp_anc.groupby(['geo', 'year', 'ceparema'], as_index=False)['inv_corp_anc'].sum(min_count=1)

# Convert all investment units from MIO EUR to absolute EUR
for df, col in [(df_gov, 'inv_gov'), (df_corp_spec, 'inv_corp_spec'), (df_corp_anc, 'inv_corp_anc')]:
    df[col] = (df[col] * 1_000_000).round().astype('Int64')

# Merge the three investment streams into ONE Fact Table
df_invest = pd.merge(df_gov[['geo', 'year', 'ceparema', 'inv_gov']], 
                     df_corp_spec[['geo', 'year', 'ceparema', 'inv_corp_spec']], 
                     on=['geo', 'year', 'ceparema'], how='outer')

df_invest = pd.merge(df_invest, 
                     df_corp_anc[['geo', 'year', 'ceparema', 'inv_corp_anc']], 
                     on=['geo', 'year', 'ceparema'], how='outer')

df_invest.rename(columns={'geo': 'country_code', 'ceparema': 'ceparema_code'}, inplace=True)
df_invest.dropna(subset=['country_code', 'year', 'ceparema_code'], inplace=True)

In [56]:
# Create dimension tables

# Fetch descriptive labels
geo_labels = eurostat.get_dic('demo_pjan', par='geo', frmt='dict')
cat_labels = eurostat.get_dic('env_ac_epigg1', par='ceparema', frmt='dict')

# Build Country Table
unique_countries = pd.concat([df_macro['country_code'], df_invest['country_code']]).unique()
df_country = pd.DataFrame({'country_code': unique_countries})
df_country['country_name'] = df_country['country_code'].map(geo_labels)
df_country.dropna(subset=['country_code'], inplace=True)

# Build Activity Table
unique_activities = df_invest['ceparema_code'].unique()
df_activity = pd.DataFrame({'ceparema_code': unique_activities})
df_activity['activity_name'] = df_activity['ceparema_code'].map(cat_labels)
df_activity.dropna(subset=['ceparema_code'], inplace=True)

In [57]:
# Summary output
print("--- DBRepo Tables Successfully Prepared ---")
print(f"Country Table:                 {len(df_country)} rows")
print(f"Environmental_Activity Table:  {len(df_activity)} rows")
print(f"Macroeconomic_Indicator Table: {len(df_macro)} rows")
print(f"Environmental_Investment Table:{len(df_invest)} rows")

--- DBRepo Tables Successfully Prepared ---
Country Table:                 56 rows
Environmental_Activity Table:  12 rows
Macroeconomic_Indicator Table: 504 rows
Environmental_Investment Table:3141 rows


### Upload data with the dbrepo client

Not yet executed!

In [ ]:
# # Country Dimension Table
# COUNTRY_TABLE_ID = "188f55be-49b0-4a8b-ad40-6c07850cb1d5"

# # Expected column order for DBRepo
# expected_country_order = ['country_code', 'country_name']

# # Check current order
# current_order = list(df_country.columns)

# if current_order != expected_country_order:
#     print("Column order mismatch detected. Reordering dataframe...")
#     df_country = df_country[expected_country_order]
# else:
#     print("Column order already correct. No changes made.")

# # Upload to DBRepo
# try:  
#     client.import_table_data(
#         database_id=DATABASE_ID, 
#         table_id=COUNTRY_TABLE_ID, 
#         dataframe=df_country
#     )
#     print(f"Success! {len(df_country)} records were successfully uploaded to the Country table.")

# except Exception as e:
#     print(f"Failed to upload Country data. Error: {e}")

In [ ]:
# # Environmental_Activity Dimension Table
# ACTIVITY_TABLE_ID = "c63043c3-4390-4811-9ceb-fa77747b55e8"

# # Expected column order for DBRepo
# expected_activity_order = ['ceparema_code', 'activity_name']

# # Check current order
# current_order = list(df_activity.columns)

# if current_order != expected_activity_order:
#     print("Column order mismatch detected. Reordering dataframe...")
#     df_activity = df_activity[expected_activity_order]
# else:
#     print("Column order already correct. No changes made.")

# # Upload to DBRepo
# try:
#     client.import_table_data(
#         database_id=DATABASE_ID,
#         table_id=ACTIVITY_TABLE_ID,
#         dataframe=df_activity
#     )
#     print(f"Success! {len(df_activity)} records were successfully uploaded to the Environmental_Activity table.")

# except Exception as e:
#     print(f"Failed to upload Environmental_Activity data. Error: {e}")

In [ ]:
# # Macroeconomic_Indicator Fact Table
# MACRO_TABLE_ID = "3d092a1b-4c9d-4bf7-91c2-e85277666082"

# # Expected column order for DBRepo
# expected_macro_order = ['country_code', 'year', 'population', 'gdp_per_capita']

# # Check current order
# current_order = list(df_macro.columns)

# if current_order != expected_macro_order:
#     print("Column order mismatch detected. Reordering dataframe...")
#     df_macro = df_macro[expected_macro_order]
# else:
#     print("Column order already correct. No changes made.")

# # Upload to DBRepo
# try:
#     client.import_table_data(
#         database_id=DATABASE_ID,
#         table_id=MACRO_TABLE_ID,
#         dataframe=df_macro
#     )
#     print(f"Success! {len(df_macro)} records were successfully uploaded to the Macroeconomic_Indicator table.")

# except Exception as e:
#     print(f"Failed to upload Macroeconomic_Indicator data. Error: {e}")

In [ ]:
# # Environmental_Investment Fact Table
# INVESTMENT_TABLE_ID = "c637f536-d50f-4ccb-bb02-d865e29d96e1"

# # Expected column order for DBRepo
# expected_investment_order = ['country_code', 'year', 'ceparema_code', 'inv_gov', 'inv_corp_spec', 'inv_corp_anc']

# # Check current order
# current_order = list(df_invest.columns)

# if current_order != expected_investment_order:
#     print("Column order mismatch detected. Reordering dataframe...")
#     df_invest = df_invest[expected_investment_order]
# else:
#     print("Column order already correct. No changes made.")

# # Upload to DBRepo
# try:
#     client.import_table_data(
#         database_id=DATABASE_ID,
#         table_id=INVESTMENT_TABLE_ID,
#         dataframe=df_invest
#     )
#     print(f"Success! {len(df_invest)} records were successfully uploaded to the Environmental_Investment table.")

# except Exception as e:
#     print(f"Failed to upload Environmental_Investment data. Error: {e}")